# Gradient Boosting Deep-Dive — XGBoost vs LightGBM vs CatBoost

The **#1 algorithms in industry and Kaggle** competitions:
- **How Gradient Boosting works** — step-by-step intuition
- **XGBoost**: Regularized gradient boosting
- **LightGBM**: Histogram-based, leaf-wise growth
- **CatBoost**: Native categorical feature handling
- **Performance Comparison** on the same dataset
- **Feature Importance** comparison across all three

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

# Install if not available
try:
    import xgboost as xgb
    print(f'XGBoost: {xgb.__version__}')
except ImportError:
    print('Install: pip install xgboost')

try:
    import lightgbm as lgb
    print(f'LightGBM: {lgb.__version__}')
except ImportError:
    print('Install: pip install lightgbm')

try:
    import catboost as cb
    print(f'CatBoost: {cb.__version__}')
except ImportError:
    print('Install: pip install catboost')

import warnings
warnings.filterwarnings('ignore')

## 1. Gradient Boosting Intuition

Gradient Boosting builds trees **sequentially**, each correcting the errors of the previous:

```
F₀(x) = initial prediction (e.g., mean)
F₁(x) = F₀(x) + η·h₁(x)    ← h₁ fits residuals of F₀
F₂(x) = F₁(x) + η·h₂(x)    ← h₂ fits residuals of F₁
...
Fₘ(x) = Fₘ₋₁(x) + η·hₘ(x)  ← each tree reduces error further
```

**Key Hyperparameters:**
- `n_estimators` / `num_boost_round`: Number of trees
- `learning_rate` (η): Step size (smaller = more conservative)
- `max_depth`: Tree depth (controls complexity)
- `subsample`: Fraction of data used per tree (stochastic gradient boosting)

### Key Differences:

| Feature | XGBoost | LightGBM | CatBoost |
|---------|---------|----------|----------|
| Tree Growth | Level-wise | **Leaf-wise** (faster) | Symmetric (balanced) |
| Speed | Fast | **Fastest** | Medium |
| Categorical | Manual encoding | Native (`categorical_feature`) | **Best native support** |
| Regularization | L1 + L2 | L1 + L2 | Built-in ordered boosting |
| Missing Values | Native handling | Native handling | Native handling |
| GPU Support | Yes | Yes | **Best GPU support** |

In [ ]:
# Load dataset
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Target range: [{y.min():.2f}, {y.max():.2f}]')

## 2. Sklearn Gradient Boosting (Baseline)

In [ ]:
# Scikit-learn GradientBoosting
start = time.time()
gb_sklearn = GradientBoostingRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gb_sklearn.fit(X_train, y_train)
sklearn_time = time.time() - start
sklearn_pred = gb_sklearn.predict(X_test)
print(f'Sklearn GB:  R²={r2_score(y_test, sklearn_pred):.4f} | '
      f'RMSE={np.sqrt(mean_squared_error(y_test, sklearn_pred)):.4f} | '
      f'Time={sklearn_time:.2f}s')

## 3. XGBoost

In [ ]:
start = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1,  # L1 and L2 regularization
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_time = time.time() - start
xgb_pred = xgb_model.predict(X_test)
print(f'XGBoost:     R²={r2_score(y_test, xgb_pred):.4f} | '
      f'RMSE={np.sqrt(mean_squared_error(y_test, xgb_pred)):.4f} | '
      f'Time={xgb_time:.2f}s')

## 4. LightGBM

In [ ]:
start = time.time()
lgb_model = lgb.LGBMRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
lgb_time = time.time() - start
lgb_pred = lgb_model.predict(X_test)
print(f'LightGBM:    R²={r2_score(y_test, lgb_pred):.4f} | '
      f'RMSE={np.sqrt(mean_squared_error(y_test, lgb_pred)):.4f} | '
      f'Time={lgb_time:.2f}s')

## 5. CatBoost

In [ ]:
start = time.time()
cb_model = cb.CatBoostRegressor(
    iterations=500, depth=5, learning_rate=0.1,
    subsample=0.8, l2_leaf_reg=3,
    random_state=42, verbose=0
)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test))
cb_time = time.time() - start
cb_pred = cb_model.predict(X_test)
print(f'CatBoost:    R²={r2_score(y_test, cb_pred):.4f} | '
      f'RMSE={np.sqrt(mean_squared_error(y_test, cb_pred)):.4f} | '
      f'Time={cb_time:.2f}s')

## 6. Head-to-Head Comparison

In [ ]:
# Comparison DataFrame
results = pd.DataFrame({
    'Model': ['Sklearn GB', 'XGBoost', 'LightGBM', 'CatBoost'],
    'R²': [r2_score(y_test, p) for p in [sklearn_pred, xgb_pred, lgb_pred, cb_pred]],
    'RMSE': [np.sqrt(mean_squared_error(y_test, p)) for p in [sklearn_pred, xgb_pred, lgb_pred, cb_pred]],
    'MAE': [mean_absolute_error(y_test, p) for p in [sklearn_pred, xgb_pred, lgb_pred, cb_pred]],
    'Train Time (s)': [sklearn_time, xgb_time, lgb_time, cb_time]
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e67e22']

for ax, metric in zip(axes, ['R²', 'RMSE', 'Train Time (s)']):
    bars = ax.bar(results['Model'], results[metric], color=colors, edgecolor='black')
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}' if metric != 'Train Time (s)' else f'{val:.2f}s',
                ha='center', fontsize=10, fontweight='bold')
    ax.set_title(metric, fontsize=13)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Gradient Boosting Libraries — Head-to-Head Comparison', fontsize=14)
plt.tight_layout()
plt.show()
results

## 7. Feature Importance Comparison

In [ ]:
# Feature importance from all three
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
models_dict = {'XGBoost': xgb_model, 'LightGBM': lgb_model, 'CatBoost': cb_model}
colors_fi = ['#3498db', '#2ecc71', '#e67e22']

for ax, (name, model), color in zip(axes, models_dict.items(), colors_fi):
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)
    ax.barh(np.array(feature_names)[sorted_idx], importances[sorted_idx], color=color)
    ax.set_title(f'{name} Feature Importance')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance — XGBoost vs LightGBM vs CatBoost', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Early Stopping — Prevent Overfitting

In [ ]:
# XGBoost with early stopping
xgb_es = xgb.XGBRegressor(n_estimators=2000, max_depth=5, learning_rate=0.05,
                            subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
xgb_es.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)],
           verbose=False)

# Plot learning curves
evals = xgb_es.evals_result()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(evals['validation_0']['rmse'], label='Train RMSE', alpha=0.7)
ax.plot(evals['validation_1']['rmse'], label='Test RMSE', alpha=0.7)
best_iter = np.argmin(evals['validation_1']['rmse'])
ax.axvline(best_iter, color='red', linestyle='--', label=f'Best Iteration: {best_iter}')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE')
ax.set_title('XGBoost Learning Curve — Early Stopping')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. When to Use Which?

| Scenario | Best Choice | Reason |
|----------|-------------|--------|
| **Large dataset (>100K rows)** | LightGBM | Fastest training, histogram-based |
| **Many categorical features** | CatBoost | Best native categorical handling |
| **General purpose / Kaggle** | XGBoost | Most mature, well-documented |
| **High cardinality categories** | CatBoost | Ordered target statistics |
| **GPU training needed** | CatBoost or XGBoost | Excellent GPU support |
| **Small dataset** | Any (CatBoost has least overfitting) | CatBoost's ordered boosting helps |
| **Interpretability needed** | XGBoost/LightGBM | Better SHAP integration |

### Pro Tip
In practice, **try all three** and pick the best. The performance differences are often small — the real differentiator is usually **feature engineering**, not the algorithm.